In [1]:
%pip install pandas numpy matplotlib seaborn requests openpyxl scikit-learn --quiet

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# requests é a biblioteca que faz requisições HTTP em Python
# É como um "navegador programático"
%pip install requests --quiet

print("✅ Biblioteca instalada!")

Note: you may need to restart the kernel to use updated packages.
✅ Biblioteca instalada!



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
# Vamos montar a URL manualmente para você entender cada parte

codigo_serie = 21082          # número da série no BCB
data_inicio  = "01/01/2015"   # de quando queremos os dados
data_fim     = "01/12/2024"   # até quando queremos

url = (
    f"https://api.bcb.gov.br/dados/serie/"
    f"bcdata.sgs.{codigo_serie}/dados"
    f"?formato=json"
    f"&dataInicial={data_inicio}"
    f"&dataFinal={data_fim}"
)

print("URL que será chamada:")
print(url)
print()
print("Cole essa URL no navegador para ver o que ela retorna!")

URL que será chamada:
https://api.bcb.gov.br/dados/serie/bcdata.sgs.21082/dados?formato=json&dataInicial=01/01/2015&dataFinal=01/12/2024

Cole essa URL no navegador para ver o que ela retorna!


In [4]:
import requests

# Fazemos a requisição — é como clicar em "Enter" no navegador
response = requests.get(url, timeout=15)

# Verificar se deu certo
# Código 200 = sucesso (como o "200 OK" do HTTP)
print(f"Status da requisição: {response.status_code}")

if response.status_code == 200:
    print("✅ Conexão com o BCB bem-sucedida!")
else:
    print("❌ Algo deu errado. Verifique sua internet.")

Status da requisição: 200
✅ Conexão com o BCB bem-sucedida!


In [5]:
# O BCB devolve os dados em formato JSON
# JSON é um formato de texto estruturado — parecido com um dicionário Python
dados_json = response.json()

# Ver os primeiros 3 registros para entender a estrutura
print("Estrutura dos dados recebidos:")
print(type(dados_json))       # vai mostrar: <class 'list'>
print()
print("Primeiros 3 registros:")
for registro in dados_json[:3]:
    print(registro)


Estrutura dos dados recebidos:
<class 'list'>

Primeiros 3 registros:
{'data': '01/01/2015', 'valor': '2.82'}
{'data': '01/02/2015', 'valor': '2.85'}
{'data': '01/03/2015', 'valor': '2.82'}


In [6]:
import pandas as pd

# Transforma a lista de dicionários em uma tabela (DataFrame)
df_inadimplencia = pd.DataFrame(dados_json)

# Converter os tipos de dados corretamente
df_inadimplencia['data']  = pd.to_datetime(df_inadimplencia['data'], format='%d/%m/%Y')
df_inadimplencia['valor'] = pd.to_numeric(df_inadimplencia['valor'], errors='coerce')

# Renomear a coluna valor para o nome da série
df_inadimplencia.rename(columns={'valor': 'Inadimplencia_Pct'}, inplace=True)

print(f"✅ DataFrame criado: {len(df_inadimplencia)} linhas")
print()
df_inadimplencia.head()

✅ DataFrame criado: 120 linhas



,data,Inadimplencia_Pct
0,2015-01-01,2.82
1,2015-02-01,2.85
2,2015-03-01,2.82
3,2015-04-01,2.96
4,2015-05-01,3.02


In [7]:
# Agora que você entendeu o processo, vamos criar uma função
# para não repetir o mesmo código 3 vezes

def baixar_serie_bcb(codigo, nome):
    """
    Baixa uma série do Banco Central e retorna um DataFrame.
    
    codigo : número da série (ex: 21082)
    nome   : nome que você quer dar à coluna de valores
    """
    url = (
        f"https://api.bcb.gov.br/dados/serie/"
        f"bcdata.sgs.{codigo}/dados"
        f"?formato=json"
        f"&dataInicial=01/01/2015"
        f"&dataFinal=01/12/2024"
    )
    
    print(f"Baixando série {codigo} ({nome})...")
    
    response = requests.get(url, timeout=15)
    
    if response.status_code != 200:
        print(f"  ❌ Erro: status {response.status_code}")
        return None
    
    dados = response.json()
    df = pd.DataFrame(dados)
    df['data']  = pd.to_datetime(df['data'], format='%d/%m/%Y')
    df['valor'] = pd.to_numeric(df['valor'], errors='coerce')
    df.rename(columns={'valor': nome}, inplace=True)
    
    print(f"  ✅ {len(df)} registros baixados!")
    return df

# Baixar as 3 séries
inadimplencia = baixar_serie_bcb(21082, 'Inadimplencia_Pct')
concessoes    = baixar_serie_bcb(20714, 'Concessoes_Credito_Mi')
spread        = baixar_serie_bcb(25433, 'Spread_Medio_Pct')

Baixando série 21082 (Inadimplencia_Pct)...
  ✅ 120 registros baixados!
Baixando série 20714 (Concessoes_Credito_Mi)...
  ✅ 120 registros baixados!
Baixando série 25433 (Spread_Medio_Pct)...
  ✅ 120 registros baixados!


In [8]:
# merge() une dois DataFrames pela coluna 'data'
# how='outer' mantém todas as datas, mesmo se uma série não tiver valor naquele mês

bcb = inadimplencia \
        .merge(concessoes, on='data', how='outer') \
        .merge(spread,     on='data', how='outer')

bcb.sort_values('data', inplace=True)
bcb.reset_index(drop=True, inplace=True)

print(f"✅ Dataset BCB consolidado!")
print(f"   {bcb.shape[0]} meses  x  {bcb.shape[1]} colunas")
print()
bcb.tail()

✅ Dataset BCB consolidado!
   120 meses  x  4 colunas



,data,Inadimplencia_Pct,Concessoes_Credito_Mi,Spread_Medio_Pct
115,2024-08-01,3.21,27.59,2.05
116,2024-09-01,3.22,27.48,2.04
117,2024-10-01,3.17,27.93,2.07
118,2024-11-01,3.14,28.44,2.11
119,2024-12-01,2.95,28.50,2.11


In [9]:
import os

# Criar pasta de output se não existir
os.makedirs('output', exist_ok=True)

# Salvar
# encoding='utf-8-sig' garante que acentos aparecem corretamente no Excel e Power BI
bcb.to_csv('output/bcb_macro.csv', index=False, encoding='utf-8-sig')

print("✅ Arquivo salvo em: output/bcb_macro.csv")
print(f"   {bcb.shape[0]} linhas  |  {bcb.shape[1]} colunas")
print()
print("Colunas do arquivo:")
for col in bcb.columns:
    print(f"  - {col}")
    

✅ Arquivo salvo em: output/bcb_macro.csv
   120 linhas  |  4 colunas

Colunas do arquivo:
  - data
  - Inadimplencia_Pct
  - Concessoes_Credito_Mi
  - Spread_Medio_Pct


In [10]:
import zipfile
import os
import shutil

# Caminho do arquivo zip
caminho_zip = 'GiveMeSomeCredit.zip'

# Criar pasta data/ se não existir
os.makedirs('data', exist_ok=True)

# Ver o que tem dentro do zip
with zipfile.ZipFile(caminho_zip, 'r') as z:
    print('Arquivos dentro do .zip:')
    for arquivo in z.namelist():
        print(f'  - {arquivo}')

Arquivos dentro do .zip:
  - Data Dictionary.xls
  - cs-test.csv
  - cs-training.csv
  - sampleEntry.csv


In [11]:
# Extrair apenas o cs-training.csv para a pasta data/
with zipfile.ZipFile(caminho_zip, 'r') as z:
    z.extract('cs-training.csv', path='data/')

print('✅ Arquivo extraído com sucesso!')

# Confirmar que o arquivo existe
tamanho = os.path.getsize('data/cs-training.csv') / (1024 * 1024)
print(f'   Localização : data/cs-training.csv')
print(f'   Tamanho     : {tamanho:.1f} MB')

✅ Arquivo extraído com sucesso!
   Localização : data/cs-training.csv
   Tamanho     : 7.2 MB


In [12]:
import pandas as pd

# Carregar o dataset
df = pd.read_csv('data/cs-training.csv', index_col=0)

print(f'✅ Dataset carregado!')
print(f'   Linhas  : {df.shape[0]:,}')
print(f'   Colunas : {df.shape[1]}')
print()
print('Colunas originais:')
for col in df.columns:
    print(f'  - {col}')

✅ Dataset carregado!
   Linhas  : 150,000
   Colunas : 11

Colunas originais:
  - SeriousDlqin2yrs
  - RevolvingUtilizationOfUnsecuredLines
  - age
  - NumberOfTime30-59DaysPastDueNotWorse
  - DebtRatio
  - MonthlyIncome
  - NumberOfOpenCreditLinesAndLoans
  - NumberOfTimes90DaysLate
  - NumberRealEstateLoansOrLines
  - NumberOfTime60-89DaysPastDueNotWorse
  - NumberOfDependents


In [13]:
colunas_pt = {
    'SeriousDelinquincy2yrs'                : 'Inadimplente',
    'RevolvingUtilizationOfUnsecuredLines'  : 'Utilizacao_Credito',
    'age'                                   : 'Idade',
    'NumberOfTime30-59DaysPastDueNotWorse'  : 'Atrasos_30_59dias',
    'DebtRatio'                             : 'Razao_Divida',
    'MonthlyIncome'                         : 'Renda_Mensal',
    'NumberOfOpenCreditLinesAndLoans'       : 'Linhas_Credito_Abertas',
    'NumberOfTimes90DaysLate'               : 'Atrasos_90dias',
    'NumberRealEstateLoansOrLines'          : 'Emprestimos_Imoveis',
    'NumberOfTime60-89DaysPastDueNotWorse'  : 'Atrasos_60_89dias',
    'NumberOfDependents'                    : 'Dependentes'
}

df.rename(columns=colunas_pt, inplace=True)

print('✅ Colunas renomeadas com sucesso!')
print()
print('Novas colunas:')
for col in df.columns:
    print(f'  - {col}')

✅ Colunas renomeadas com sucesso!

Novas colunas:
  - SeriousDlqin2yrs
  - Utilizacao_Credito
  - Idade
  - Atrasos_30_59dias
  - Razao_Divida
  - Renda_Mensal
  - Linhas_Credito_Abertas
  - Atrasos_90dias
  - Emprestimos_Imoveis
  - Atrasos_60_89dias
  - Dependentes


In [14]:
# rename() → renomeia colunas do DataFrame
# columns= → recebe um dicionário {'nome_antigo': 'nome_novo'}
# Aqui corrigimos apenas a coluna que ficou com nome errado
df.rename(columns={'SeriousDlqin2yrs': 'Inadimplente'}, inplace=True)
# inplace=True → modifica o df original direto, sem precisar escrever:
# df = df.rename(...) 

print('✅ Corrigido!')

# df.columns → lista com os nomes de todas as colunas
# O for percorre cada nome e o print exibe um por linha
print('Novas colunas:')
for col in df.columns:
    print(f'  - {col}')  # f'...' → f-string, {col} é substituído pelo nome real

✅ Corrigido!
Novas colunas:
  - Inadimplente
  - Utilizacao_Credito
  - Idade
  - Atrasos_30_59dias
  - Razao_Divida
  - Renda_Mensal
  - Linhas_Credito_Abertas
  - Atrasos_90dias
  - Emprestimos_Imoveis
  - Atrasos_60_89dias
  - Dependentes


In [15]:
# shape → retorna uma tupla (linhas, colunas)
# shape[0] = linhas | shape[1] = colunas
print(f'Linhas  : {df.shape[0]:,}')  # :, formata o número com separador de milhar
print(f'Colunas : {df.shape[1]}')
print()

# Montamos um DataFrame resumo para ver informações de cada coluna
# isnull() → retorna True/False para cada célula (True = é nulo)
# sum()    → soma os True (True = 1, False = 0) → total de nulos
# round(2) → arredonda para 2 casas decimais
resumo = pd.DataFrame({
    'Tipo'    : df.dtypes,
    'Nulos'   : df.isnull().sum(),
    'Nulos_%' : (df.isnull().sum() / len(df) * 100).round(2),
    'Unicos'  : df.nunique(),  # nunique() → conta valores distintos
})

print(resumo.to_string())  # to_string() → exibe a tabela completa sem cortar


Linhas  : 150,000
Colunas : 11

                           Tipo  Nulos  Nulos_%  Unicos
Inadimplente              int64      0     0.00       2
Utilizacao_Credito      float64      0     0.00  125728
Idade                     int64      0     0.00      86
Atrasos_30_59dias         int64      0     0.00      16
Razao_Divida            float64      0     0.00  114194
Renda_Mensal            float64  29731    19.82   13594
Linhas_Credito_Abertas    int64      0     0.00      58
Atrasos_90dias            int64      0     0.00      19
Emprestimos_Imoveis       int64      0     0.00      28
Atrasos_60_89dias         int64      0     0.00      13
Dependentes             float64   3924     2.62      13


In [16]:
# describe() → calcula estatísticas para todas as colunas numéricas
# count  = total de valores não nulos (note que Renda_Mensal terá menos que 150.000)
# mean   = média aritmética
# std    = desvio padrão (o quanto os valores variam em torno da média)
# min    = menor valor encontrado
# 25%    = primeiro quartil (25% dos dados estão abaixo desse valor)
# 50%    = mediana (valor do meio — mais robusta que a média)
# 75%    = terceiro quartil (75% dos dados estão abaixo desse valor)
# max    = maior valor encontrado
df.describe().round(2)

,Inadimplente,Utilizacao_Credito,Idade,Atrasos_30_59dias,Razao_Divida,Renda_Mensal,Linhas_Credito_Abertas,Atrasos_90dias,Emprestimos_Imoveis,Atrasos_60_89dias,Dependentes
count,150000.00,150000.00,150000.00,150000.00,150000.00,120269.00,150000.00,150000.00,150000.00,150000.00,146076.00
mean,0.07,6.05,52.30,0.42,353.01,6670.22,8.45,0.27,1.02,0.24,0.76
std,0.25,249.76,14.77,4.19,2037.82,14384.67,5.15,4.17,1.13,4.16,1.12
min,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
25%,0.00,0.03,41.00,0.00,0.18,3400.00,5.00,0.00,0.00,0.00,0.00
50%,0.00,0.15,52.00,0.00,0.37,5400.00,8.00,0.00,1.00,0.00,0.00
75%,0.00,0.56,63.00,0.00,0.87,8249.00,11.00,0.00,2.00,0.00,1.00
max,1.00,50708.00,109.00,98.00,329664.00,3008750.00,58.00,98.00,54.00,98.00,20.00


In [17]:
# SEMPRE trabalhe em uma copia — preserva o original intacto
# Se algo der errado na limpeza, o df original continua disponivel
# sem precisar recarregar o CSV do zero
df_limpo = df.copy()

print(f'Original    : {df.shape[0]:,} linhas  — nao sera alterado')
print(f'Para limpar : {df_limpo.shape[0]:,} linhas')

Original    : 150,000 linhas  — nao sera alterado
Para limpar : 150,000 linhas


In [18]:
# Guardar quantidade antes para comparar depois
antes = len(df_limpo)

# df[condicao]: retorna apenas as linhas onde a condicao e True
# df_limpo['Idade'] > 0: True para idades validas, False para Idade = 0
df_limpo = df_limpo[df_limpo['Idade'] > 0]

removidas = antes - len(df_limpo)
print(f'Linhas removidas (Idade = 0): {removidas}')
print(f'Dataset atual: {df_limpo.shape[0]:,} linhas')

Linhas removidas (Idade = 0): 1
Dataset atual: 149,999 linhas


In [19]:
# Guardar quantidade de nulos antes para confirmar depois
nulos_antes = df_limpo['Renda_Mensal'].isnull().sum()

# PASSO 1: Criar coluna temporaria de faixa etaria para agrupar
# pd.cut(): divide valores numericos em intervalos com labels definidos
df_limpo['_faixa'] = pd.cut(
    df_limpo['Idade'],
    bins=[0, 30, 45, 60, 110],
    labels=['18-30', '31-45', '46-60', '60+']
)

# PASSO 2: Imputar com mediana do grupo
# groupby(): agrupa por faixa etaria
# transform(): aplica a funcao e devolve no mesmo tamanho do DataFrame
# lambda x: x.fillna(x.median()) — preenche nulos com a mediana do grupo
df_limpo['Renda_Mensal'] = (
    df_limpo.groupby('_faixa', observed=True)['Renda_Mensal']
    .transform(lambda x: x.fillna(x.median()))
)

# PASSO 3: Fallback — se ainda houver nulos usa mediana geral
df_limpo['Renda_Mensal'].fillna(df_limpo['Renda_Mensal'].median(), inplace=True)

# PASSO 4: Remover coluna temporaria — nao e mais necessaria
df_limpo.drop(columns=['_faixa'], inplace=True)

nulos_depois = df_limpo['Renda_Mensal'].isnull().sum()
print(f'Nulos antes : {nulos_antes:,}')
print(f'Nulos depois: {nulos_depois}')
print(f'Resultado   : {"OK — todos imputados!" if nulos_depois == 0 else "ATENCAO — ainda ha nulos"}')

Nulos antes : 29,731
Nulos depois: 0
Resultado   : OK — todos imputados!


C:\Users\juant\AppData\Local\Temp\ipykernel_28240\1990631697.py:22: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  df_limpo['Renda_Mensal'].fillna(df_limpo['Renda_Mensal'].median(), inplace=True)


In [20]:
# Correcao do aviso ChainedAssignmentError
# Em vez de inplace=True, reatribuimos diretamente a coluna
df_limpo['Renda_Mensal'] = df_limpo['Renda_Mensal'].fillna(
    df_limpo['Renda_Mensal'].median()
)

# Confirmar que ainda esta zerado
print(f'Nulos em Renda_Mensal: {df_limpo["Renda_Mensal"].isnull().sum()}')
print('Linha 22 corrigida — sem avisos!')

Nulos em Renda_Mensal: 0
Linha 22 corrigida — sem avisos!


In [21]:
# mode(): retorna os valores mais frequentes como Series
# [0]: pega o primeiro (mais frequente)
moda = df_limpo['Dependentes'].mode()[0]

nulos_antes = df_limpo['Dependentes'].isnull().sum()

# Reatribuicao direta — sem inplace para evitar o aviso
df_limpo['Dependentes'] = df_limpo['Dependentes'].fillna(moda)

nulos_depois = df_limpo['Dependentes'].isnull().sum()
print(f'Moda de Dependentes : {moda}')
print(f'Nulos antes         : {nulos_antes:,}')
print(f'Nulos depois        : {nulos_depois}')

Moda de Dependentes : 0.0
Nulos antes         : 3,924
Nulos depois        : 0


In [22]:
# Colunas que identificamos com outliers extremos no describe()
colunas_outlier = ['Renda_Mensal', 'Utilizacao_Credito', 'Razao_Divida']

antes = len(df_limpo)

for col in colunas_outlier:
    media  = df_limpo[col].mean()   # media da coluna
    std    = df_limpo[col].std()    # desvio padrao
    limite = media + 3 * std        # limite superior: media + 3 desvios

    removidos = (df_limpo[col] > limite).sum()
    df_limpo  = df_limpo[df_limpo[col] <= limite]

    print(f'{col:<25}: {removidos:,} outliers removidos  (limite: {limite:,.2f})')

depois = len(df_limpo)
print()
print(f'Linhas antes : {antes:,}')
print(f'Linhas depois: {depois:,}')
print(f'Removidas    : {antes - depois:,}  ({(antes-depois)/antes*100:.2f}%)')

Renda_Mensal             : 353 outliers removidos  (limite: 45,112.41)
Utilizacao_Credito       : 191 outliers removidos  (limite: 756.21)
Razao_Divida             : 656 outliers removidos  (limite: 6,474.24)

Linhas antes : 149,999
Linhas depois: 148,799
Removidas    : 1,200  (0.80%)


In [23]:
print('=' * 45)
print('VERIFICACAO FINAL DA LIMPEZA')
print('=' * 45)

# Checar se ainda ha nulos em qualquer coluna
nulos = df_limpo.isnull().sum()

if nulos.sum() == 0:
    print('Nenhum nulo restante. Dataset 100% limpo!')
else:
    print('ATENCAO — nulos restantes:')
    print(nulos[nulos > 0])

print()
print(f'Dataset original : {df.shape[0]:,} linhas')
print(f'Dataset limpo    : {df_limpo.shape[0]:,} linhas')
print(f'Total removido   : {df.shape[0] - df_limpo.shape[0]:,} linhas')
print(f'Reducao          : {(df.shape[0]-df_limpo.shape[0])/df.shape[0]*100:.2f}%')
print()
print(f'Colunas          : {df_limpo.shape[1]}')
print()

# Confirmar tipos de dados
print('Tipos de dados apos limpeza:')
print(df_limpo.dtypes)

VERIFICACAO FINAL DA LIMPEZA
Nenhum nulo restante. Dataset 100% limpo!

Dataset original : 150,000 linhas
Dataset limpo    : 148,799 linhas
Total removido   : 1,201 linhas
Reducao          : 0.80%

Colunas          : 11

Tipos de dados apos limpeza:
Inadimplente                int64
Utilizacao_Credito        float64
Idade                       int64
Atrasos_30_59dias           int64
Razao_Divida              float64
Renda_Mensal              float64
Linhas_Credito_Abertas      int64
Atrasos_90dias              int64
Emprestimos_Imoveis         int64
Atrasos_60_89dias           int64
Dependentes               float64
dtype: object


In [24]:
import numpy as np

# ── FEATURE 1: Faixa Etaria ───────────────────────────────
# pd.cut(): divide valores em intervalos com labels definidos
df_limpo['Faixa_Etaria'] = pd.cut(
    df_limpo['Idade'],
    bins=[0, 30, 45, 60, 110],
    labels=['Jovem (18-30)', 'Adulto (31-45)', 'Maduro (46-60)', 'Senior (60+)']
)

# ── FEATURE 2: Nivel de Endividamento ────────────────────
# pd.qcut(): divide em quartis — 4 grupos com mesma quantidade de clientes
df_limpo['Nivel_Endividamento'] = pd.qcut(
    df_limpo['Razao_Divida'],
    q=4,
    labels=['Baixo', 'Medio', 'Alto', 'Critico'],
    duplicates='drop'  # evita erro se houver valores identicos nos limites
)

# ── FEATURE 3: Nivel de Utilizacao ───────────────────────
df_limpo['Nivel_Utilizacao'] = pd.cut(
    df_limpo['Utilizacao_Credito'],
    bins=[-0.01, 0.25, 0.50, 0.75, 1.01],
    labels=['Baixa (0-25%)', 'Moderada (25-50%)', 'Alta (50-75%)', 'Maxima (75-100%)']
)

# ── FEATURE 4: Historico de Atrasos (soma ponderada) ─────
# Atrasos mais graves recebem peso maior na soma
df_limpo['Historico_Atrasos'] = (
    df_limpo['Atrasos_30_59dias'] * 1 +  # leve: peso 1
    df_limpo['Atrasos_60_89dias'] * 2 +  # medio: peso 2
    df_limpo['Atrasos_90dias']    * 3    # grave: peso 3
)

# ── FEATURE 5: Score de Risco Composto (0 a 100) ─────────
# Normalizar: traz cada variavel para escala 0-1 antes de combinar
# Sem isso, variaveis com valores maiores dominariam o score
def normalizar(serie):
    return (serie - serie.min()) / (serie.max() - serie.min())

df_limpo['Score_Risco_Composto'] = (
    normalizar(df_limpo['Utilizacao_Credito']) * 0.40 +  # maior peso
    normalizar(df_limpo['Historico_Atrasos'])  * 0.35 +
    normalizar(df_limpo['Razao_Divida'])        * 0.25
) * 100  # escala de 0 a 100

print(f'Features criadas! Dataset agora com {df_limpo.shape[1]} colunas')
print()
print('Novas colunas adicionadas:')
novas = ['Faixa_Etaria','Nivel_Endividamento','Nivel_Utilizacao',
         'Historico_Atrasos','Score_Risco_Composto']
for col in novas:
    print(f'  - {col}: {df_limpo[col].dtype}')
print()
print('Amostra das novas colunas:')
df_limpo[novas].head()

Features criadas! Dataset agora com 16 colunas

Novas colunas adicionadas:
  - Faixa_Etaria: category
  - Nivel_Endividamento: category
  - Nivel_Utilizacao: category
  - Historico_Atrasos: int64
  - Score_Risco_Composto: float64

Amostra das novas colunas:


,Faixa_Etaria,Nivel_Endividamento,Nivel_Utilizacao,Historico_Atrasos,Score_Risco_Composto
1,Adulto (31-45),Alto,Maxima (75-100%),2,0.163176
2,Adulto (31-45),Baixo,Maxima (75-100%),0,0.051724
3,Adulto (31-45),Baixo,Alta (50-75%),4,0.273668
4,Jovem (18-30),Baixo,Baixa (0-25%),0,0.012659
6,Senior (60+),Alto,Baixa (0-25%),0,0.012867


In [25]:
# Carregar o bcb_macro.csv gerado anteriormente
bcb = pd.read_csv('output/bcb_macro.csv', parse_dates=['data'])
# parse_dates=['data']: le a coluna data ja como tipo datetime

# ══ GRUPO 1: Perfil do Cliente ══════════════════════════

# KPI 1: % de clientes que deram calote
kpi1 = df_limpo['Inadimplente'].mean() * 100

# KPI 2: Renda media por nivel de endividamento
kpi2 = df_limpo.groupby('Nivel_Endividamento', observed=True)['Renda_Mensal'].mean()

# KPI 3: % de clientes por faixa etaria
kpi3 = df_limpo['Faixa_Etaria'].value_counts(normalize=True) * 100

# KPI 4: Media de dependentes por grupo (adimplente vs inadimplente)
kpi4 = df_limpo.groupby('Inadimplente')['Dependentes'].mean()

# KPI 5: % de clientes por nivel de endividamento
kpi5 = df_limpo['Nivel_Endividamento'].value_counts(normalize=True) * 100

# ══ GRUPO 2: Comportamento de Credito ═══════════════════

# KPI 6: Media de utilizacao de credito em %
kpi6 = df_limpo['Utilizacao_Credito'].mean() * 100

# KPI 7: Media de linhas de credito abertas
kpi7 = df_limpo['Linhas_Credito_Abertas'].mean()

# KPI 8: Razao divida/renda media por faixa etaria
kpi8 = df_limpo.groupby('Faixa_Etaria', observed=True)['Razao_Divida'].mean()

# KPI 9: % de clientes com 3+ atrasos graves (90+ dias)
kpi9 = (df_limpo['Atrasos_90dias'] >= 3).mean() * 100

# KPI 10: Historico de atrasos medio por grupo
kpi10 = df_limpo.groupby('Inadimplente')['Historico_Atrasos'].mean()

# ══ GRUPO 3: Risco ══════════════════════════════════════

# KPI 11: Taxa de default por nivel de endividamento
kpi11 = df_limpo.groupby('Nivel_Endividamento', observed=True)['Inadimplente'].mean() * 100

# KPI 15: Top 5 variaveis mais correlacionadas com inadimplencia
kpi15 = (df_limpo.corr(numeric_only=True)['Inadimplente']
         .abs().drop('Inadimplente').sort_values(ascending=False).head(5))

# ══ GRUPO 4: Contexto Macro BCB ═════════════════════════

# KPI 16: Ultimo valor de inadimplencia nacional
kpi16 = bcb['Inadimplencia_Pct'].iloc[-1]

# KPI 17: Ultima concessao de credito (R$ milhoes)
kpi17 = bcb['Concessoes_Credito_Mi'].iloc[-1]

# KPI 18: Spread bancario medio historico
kpi18 = bcb['Spread_Medio_Pct'].mean()

# KPI 19: Variacao da inadimplencia no ultimo mes
kpi19 = bcb['Inadimplencia_Pct'].diff().iloc[-1]

# KPI 20: Correlacao entre spread e inadimplencia
kpi20 = bcb['Inadimplencia_Pct'].corr(bcb['Spread_Medio_Pct'])

# ══ PAINEL RESUMO ════════════════════════════════════════
print('=' * 50)
print('       PAINEL DE KPIs — RISCO DE CREDITO')
print('=' * 50)
print(f'KPI 01 | Taxa de Inadimplencia       : {kpi1:.2f}%')
print(f'KPI 06 | Utilizacao Media de Credito : {kpi6:.2f}%')
print(f'KPI 07 | Media Linhas de Credito     : {kpi7:.1f}')
print(f'KPI 09 | Clientes c/ Atraso Grave    : {kpi9:.2f}%')
print(f'KPI 16 | Inadimplencia Nacional BCB  : {kpi16:.2f}%')
print(f'KPI 17 | Concessoes BCB              : R$ {kpi17:,.0f} Mi')
print(f'KPI 18 | Spread Bancario Medio       : {kpi18:.2f}%')
print(f'KPI 19 | Variacao Mensal Inadimpl.   : {kpi19:+.3f}%')
print(f'KPI 20 | Correlacao Spread x Default : {kpi20:.3f}')
print('=' * 50)
print()
print('KPI 11 — Default por Nivel de Endividamento:')
print(kpi11.to_string())
print()
print('KPI 15 — Top 5 Variaveis Preditivas:')
print(kpi15.to_string())

       PAINEL DE KPIs — RISCO DE CREDITO
KPI 01 | Taxa de Inadimplencia       : 6.68%
KPI 06 | Utilizacao Media de Credito : 42.16%
KPI 07 | Media Linhas de Credito     : 8.4
KPI 09 | Clientes c/ Atraso Grave    : 1.03%
KPI 16 | Inadimplencia Nacional BCB  : 2.95%
KPI 17 | Concessoes BCB              : R$ 28 Mi
KPI 18 | Spread Bancario Medio       : 1.98%
KPI 19 | Variacao Mensal Inadimpl.   : -0.190%
KPI 20 | Correlacao Spread x Default : 0.769

KPI 11 — Default por Nivel de Endividamento:
Nivel_Endividamento
Baixo      6.091398
Medio      5.615591
Alto       8.180327
Critico    6.844086

KPI 15 — Top 5 Variaveis Preditivas:
Atrasos_30_59dias    0.125735
Atrasos_90dias       0.117530
Idade                0.115558
Historico_Atrasos    0.114315
Atrasos_60_89dias    0.102588


In [26]:
import os
os.makedirs('output', exist_ok=True)

# 1. Dataset principal limpo com todas as features
# encoding='utf-8-sig': garante acentos no Power BI e Excel
df_limpo.to_csv('output/credit_clean.csv', index=False, encoding='utf-8-sig')
print(f'credit_clean.csv    : {df_limpo.shape[0]:,} linhas x {df_limpo.shape[1]} colunas')

# 2. KPIs escalares em tabela para os cards do Power BI
kpis_tabela = pd.DataFrame({
    'KPI': [
        'Taxa Global de Inadimplencia (%)',
        'Utilizacao Media de Credito (%)',
        'Media de Linhas de Credito',
        'Clientes com Atraso Grave 90d (%)',
        'Inadimplencia Nacional BCB (%)',
        'Concessoes de Credito BCB (R$ Mi)',
        'Spread Bancario Medio (%)',
        'Variacao Mensal Inadimplencia (%)',
        'Correlacao Spread x Inadimplencia',
    ],
    'Valor': [
        round(kpi1,2), round(kpi6,2), round(kpi7,1),
        round(kpi9,2), round(kpi16,2), round(kpi17,0),
        round(kpi18,2), round(kpi19,3), round(kpi20,3),
    ]
})
kpis_tabela.to_csv('output/kpis_resumo.csv', index=False, encoding='utf-8-sig')
print(f'kpis_resumo.csv     : {len(kpis_tabela)} KPIs escalares')

# bcb_macro.csv ja existe na pasta output/ desde o inicio
print(f'bcb_macro.csv       : ja existente na pasta output/')

print()
print('Arquivos prontos para o Power BI:')
print('  output/credit_clean.csv')
print('  output/bcb_macro.csv')
print('  output/kpis_resumo.csv')

credit_clean.csv    : 148,799 linhas x 16 colunas
kpis_resumo.csv     : 9 KPIs escalares
bcb_macro.csv       : ja existente na pasta output/

Arquivos prontos para o Power BI:
  output/credit_clean.csv
  output/bcb_macro.csv
  output/kpis_resumo.csv
